# Recombine the Falcon outputs of split MGFs

Concatenates the Falcon representative spectra of the C24 alkamine trihydroxy parts into one MGF with sequential SCANS, keeping the original `file:scan` in `FILENAME_SCAN`.

In [ ]:
import os

# Root of the local project folder (the one holding Falcon/, Queries_final_mgf/,
# final_mgfs_library_generation/, ...). Edit this single line to run elsewhere.
PROJECT_DIR = '/media/dorrestein/Helena2/Bile_acids_update'

In [ ]:
from pathlib import Path

# Input MGF files to combine
input_mgfs = [
    f"{PROJECT_DIR}/Falcon/all_falcon_outputs/C24_alkylamine_trihydroxy_part1_falcon_eps01_minsamples2_scan.mgf",
    f"{PROJECT_DIR}/Falcon/all_falcon_outputs/C24_alkylamine_trihydroxy_part2_falcon_eps01_minsamples2_scan.mgf",
    f"{PROJECT_DIR}/Falcon/all_falcon_outputs/C24_alkylamine_trihydroxy_part3_falcon_eps01_minsamples2_scan.mgf",
    f"{PROJECT_DIR}/Falcon/all_falcon_outputs/C24_alkylamine_trihydroxy_part4_falcon_eps01_minsamples2_scan.mgf",
]

In [ ]:
def rewrite_spectrum_block(block_lines, new_scan, source_name):
    """Rewrite one BEGIN/END IONS block with a new SCANS and added FILENAME_SCAN."""
    original_scan = "UNKNOWN"
    scans_index = None

    for i, line in enumerate(block_lines):
        if line.startswith("SCANS="):
            scans_index = i
            original_scan = line.split("=", 1)[1].strip() or "UNKNOWN"
            break

    if scans_index is not None:
        rewritten = []
        for line in block_lines:
            if line.startswith("SCANS="):
                rewritten.append(f"SCANS={new_scan}\n")
                rewritten.append(f"FILENAME_SCAN={source_name}:{original_scan}\n")
            else:
                rewritten.append(line)
        return rewritten

    # If SCANS is missing, inject both fields right after BEGIN IONS.
    rewritten = [block_lines[0]]
    rewritten.append(f"SCANS={new_scan}\n")
    rewritten.append(f"FILENAME_SCAN={source_name}:{original_scan}\n")
    rewritten.extend(block_lines[1:])
    return rewritten


def combine_and_renumber_mgfs(input_files, output_file):
    scan_counter = 1
    spectra_written = 0

    with open(output_file, "w") as out_f:
        for mgf_path in input_files:
            source_name = Path(mgf_path).name
            in_block = False
            block = []

            with open(mgf_path, "r") as in_f:
                for line in in_f:
                    if line.startswith("BEGIN IONS"):
                        in_block = True
                        block = [line]
                        continue

                    if in_block:
                        block.append(line)
                        if line.startswith("END IONS"):
                            rewritten = rewrite_spectrum_block(block, scan_counter, source_name)
                            out_f.writelines(rewritten)
                            scan_counter += 1
                            spectra_written += 1
                            in_block = False
                            block = []

    return spectra_written

In [ ]:
output_mgf = f"{PROJECT_DIR}/Falcon/all_falcon_outputs/C24_alkylamine_trihydroxy_falcon_eps01_minsamples2_renumbered.mgf"

n_spectra = combine_and_renumber_mgfs(input_mgfs, output_mgf)
print(f"Wrote {n_spectra} spectra to: {output_mgf}")

In [ ]:
# Quick validation: total spectra, duplicate scan check, and a preview of FILENAME_SCAN entries
scans = []
filename_scan_preview = []

with open(output_mgf, "r") as f:
    for line in f:
        if line.startswith("SCANS="):
            scans.append(int(line.split("=", 1)[1].strip()))
        elif line.startswith("FILENAME_SCAN=") and len(filename_scan_preview) < 5:
            filename_scan_preview.append(line.strip())

print("Total SCANS lines:", len(scans))
print("Unique SCANS:", len(set(scans)))
print("Any repeated SCANS:", len(scans) != len(set(scans)))
print("First FILENAME_SCAN lines:")
for item in filename_scan_preview:
    print("  ", item)